In [3]:
#r "C:\Users\user\Desktop\Remish\practice2026\task14\bin\Debug\net10.0\task14.dll"
#r "nuget: ScottPlot, 5.0.21"

using System;
using System.Diagnostics;
using System.IO;
using System.Collections.Generic;
using System.Linq;
using task14;

double leftBound = -100.0;
double rightBound = 100.0;
Func<double, double> selectedFunction = Math.Sin;
double requiredAccuracy = 1e-4;
double referenceValue = 0.0;

double[] stepVariants = { 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6 };
int[] threadOptions = { 1, 2, 4, 6, 8, 12, 16 };
int measurementsPerRun = 5;

double optimalStep = 0;
int bestThreadCount = 0;
double minParallelTime = 0;
List<double> timeResults = new List<double>();
double maxEfficiencyGain = 0;

double ComputeIntegral(double a, double b, Func<double, double> func, double step, int threads)
{
    if (threads == 1)
        return DefiniteIntegral.SolveSingleThreaded(a, b, func, step);
    else
        return DefiniteIntegral.Solve(a, b, func, step, threads);
}

double MeasureExecutionTime(double a, double b, Func<double, double> func, double step, int threads, int runs)
{
    List<double> results = new List<double>();
    for (int attempt = 0; attempt < runs; attempt++)
    {
        Stopwatch timer = Stopwatch.StartNew();
        ComputeIntegral(a, b, func, step, threads);
        timer.Stop();
        results.Add(timer.Elapsed.TotalMilliseconds);
    }
    return results.Average();
}

double EvaluateEfficiency(double singleThreadTime, double parallelTime)
{
    return ((singleThreadTime - parallelTime) / singleThreadTime) * 100;
}

void DisplayResults(double step, int threads, double singleTime, double parallelTime, double efficiency)
{
    Console.WriteLine("РЕЗУЛЬТАТЫ ИССЛЕДОВАНИЯ");
    Console.WriteLine($"Размер шага: {step:E1}");
    Console.WriteLine($"Количество потоков: {threads}");
    Console.WriteLine($"Последовательное выполнение: {singleTime:F2} мс");
    Console.WriteLine($"Параллельное выполнение: {parallelTime:F2} мс");
    Console.WriteLine($"Ускорение: {efficiency:F2}%");
    Console.WriteLine($"Статус: {(efficiency >= 15.0 ? "ДОСТИГНУТ" : "НЕ ДОСТИГНУТ")} порог 15%");
}

void SaveReport(string filePath, double step, int threads, double singleTime, double parallelTime, double efficiency, double left, double right)
{
    using (StreamWriter writer = new StreamWriter(filePath, false, System.Text.Encoding.UTF8))
    {
        writer.WriteLine("ОТЧЁТ ОПТИМИЗАЦИИ ПАРАЛЛЕЛЬНЫХ ВЫЧИСЛЕНИЙ");
        writer.WriteLine($"Дата: {DateTime.Now:yyyy-MM-dd HH:mm:ss}");
        writer.WriteLine($"Функция: sin(x)");
        writer.WriteLine($"Интервал: [{left}, {right}]");
        writer.WriteLine($"Параметры вычислений:");
        writer.WriteLine($"  Шаг: {step:E1} ({(int)((right - left) / step):N0} итераций)");
        writer.WriteLine($"  Число потоков: {threads}");
        writer.WriteLine($"Производительность:");
        writer.WriteLine($"  Однопоточный режим: {singleTime:F2} мс");
        writer.WriteLine($"  Многопоточный режим: {parallelTime:F2} мс");
        writer.WriteLine($"  Прирост: {efficiency:F2}%");
        writer.WriteLine($"  Критерий 15%: {(efficiency >= 15.0 ? "ВЫПОЛНЕН" : "НЕ ВЫПОЛНЕН")}");
    }
}

void GenerateChart(List<double> times, int[] threads, double step)
{
    var chart = new ScottPlot.Plot();
    double[] xData = times.ToArray();
    double[] yData = threads.Select(t => (double)t).ToArray();
    
    var scatterPlot = chart.Add.Scatter(xData, yData);
    scatterPlot.LineWidth = 3;
    scatterPlot.MarkerSize = 10;
    scatterPlot.Color = ScottPlot.Color.FromHex("#1e3a8a");
    
    chart.Title($"Производительность вычислений (шаг {step:E1})");
    chart.XLabel("Время выполнения (мс)");
    chart.YLabel("Количество потоков");
    
    string timeStamp = DateTime.Now.Ticks.ToString();
    string fileName = $"performance_analysis_{step:E1}_{timeStamp}.png";
    
    if (File.Exists(fileName))
        File.Delete(fileName);
    
    chart.SavePng(fileName, 700, 500);
    Console.WriteLine($"\nГрафик сохранён: {fileName}");
    
    if (File.Exists(fileName))
        display(HTML($"<img src='{fileName}?t={timeStamp}' width='700'/>"));
}

foreach (var currentStep in stepVariants)
{
    double computedValue = DefiniteIntegral.SolveSingleThreaded(leftBound, rightBound, selectedFunction, currentStep);
    double calculationError = Math.Abs(computedValue - referenceValue);
    Console.WriteLine($"\nАнализ шага {currentStep:E1}");
    Console.WriteLine($"Погрешность: {calculationError:E2} (требуется < {requiredAccuracy:E1})");
    
    if (calculationError > requiredAccuracy)
    {
        Console.WriteLine($"Шаг {currentStep:E1} не обеспечивает требуемую точность, исключается из рассмотрения");
        continue;
    }
    
    List<double> measuredTimes = new List<double>();
    
    foreach (int threadCount in threadOptions)
    {
        double avgTime = MeasureExecutionTime(leftBound, rightBound, selectedFunction, currentStep, threadCount, measurementsPerRun);
        measuredTimes.Add(avgTime);
        Console.WriteLine($"  Потоков: {threadCount,2} | Время: {avgTime,6:F2} мс");
    }
    
    double singleThreadTime = measuredTimes[0];
    double bestParallelResult = measuredTimes.Skip(1).Min();
    double efficiency = EvaluateEfficiency(singleThreadTime, bestParallelResult);
    Console.WriteLine($"  Эффективность: {efficiency:F2}%");
    
    if (efficiency > 15.0)
    {
        optimalStep = currentStep;
        bestThreadCount = threadOptions[measuredTimes.IndexOf(bestParallelResult)];
        minParallelTime = bestParallelResult;
        timeResults = measuredTimes;
        maxEfficiencyGain = efficiency;
        Console.WriteLine($"  Оптимальный шаг: {currentStep:E1} (эффективность {efficiency:F2}%)");
        break;
    }
    else
    {
        Console.WriteLine($"  Шаг {currentStep:E1} отклонён (эффективность {efficiency:F2}% < 15%)");
    }
}

if (optimalStep == 0)
{
    Console.WriteLine("\nОШИБКА: Не удалось найти подходящий шаг для оптимизации!");
    return;
}

double baseTime = timeResults[0];


Console.WriteLine("ИТОГОВЫЕ ПАРАМЕТРЫ ОПТИМИЗАЦИИ");
Console.WriteLine($"Оптимальный шаг: {optimalStep:E1}");
Console.WriteLine($"Количество потоков: {bestThreadCount}");
Console.WriteLine($"Однопоточный режим: {baseTime,6:F2} мс");
Console.WriteLine($"Многопоточный режим: {minParallelTime,6:F2} мс");
Console.WriteLine($"Ускорение: {maxEfficiencyGain,6:F2}%");
Console.WriteLine($"Порог 15%: {(maxEfficiencyGain >= 15.0 ? "ПРЕВЫШЕН" : "НЕ ДОСТИГНУТ")}");

string reportFile = "optimization_report.txt";
SaveReport(reportFile, optimalStep, bestThreadCount, baseTime, minParallelTime, maxEfficiencyGain, leftBound, rightBound);
Console.WriteLine($"\nОтчёт сохранён: {reportFile}");

GenerateChart(timeResults, threadOptions, optimalStep);

Installed Packages ScottPlot, 5.0.21


Анализ шага 1.0E-001
Погрешность: 5.04E-015 (требуется < 1.0E-004)
  Потоков:  1 | Время:   0.10 мс
  Потоков:  2 | Время:   0.48 мс
  Потоков:  4 | Время:   1.00 мс
  Потоков:  6 | Время:   1.25 мс
  Потоков:  8 | Время:   1.62 мс
  Потоков: 12 | Время:   1.66 мс
  Потоков: 16 | Время:  16.30 мс
  Эффективность: -395.77%
  Шаг 1.0E-001 отклонён (эффективность -395.77% < 15%)

Анализ шага 1.0E-002
Погрешность: 1.49E-014 (требуется < 1.0E-004)
  Потоков:  1 | Время:   0.85 мс
  Потоков:  2 | Время:   2.82 мс
  Потоков:  4 | Время:   1.13 мс
  Потоков:  6 | Время:   1.56 мс
  Потоков:  8 | Время:   1.63 мс
  Потоков: 12 | Время:   2.28 мс
  Потоков: 16 | Время:   3.04 мс
  Эффективность: -32.76%
  Шаг 1.0E-002 отклонён (эффективность -32.76% < 15%)

Анализ шага 1.0E-003
Погрешность: 2.34E-016 (требуется < 1.0E-004)
  Потоков:  1 | Время:   6.91 мс
  Потоков:  2 | Время:   4.33 мс
  Потоков:  4 | Время:   3.65 мс
  Потоков:  6 | Время:   3.84 мс
  Потоков:  8 | Время:   4.42 мс
  Потоков

<null>